# SigLIP Relevance Playground

특정 dataset `Id`에 대해 SigLIP으로 다음 점수를 확인합니다.

- `image_sentence_siglip_prob`: 원본 이미지 프레임과 `Sentence`의 SigLIP image-text probability
- `caption_sentence_text_cosine`: 생성된 caption과 `Sentence`의 SigLIP text embedding cosine similarity
- `combined_relevance`: 필터링에서 쓰기 좋은 결합 점수. 이미지 점수와 caption 점수가 둘 다 있으면 더 낮은 값을 사용합니다.

`sentencepiece`, `protobuf`, `transformers`, `torch`, `pillow`, `pandas`가 필요합니다.

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
SPLIT = "train"  # "train" or "test"
DATASET_ID = "u7w0lr"

SIGLIP_MODEL = "google/siglip-so400m-patch14-384"
DEVICE = "auto"  # "auto", "cpu", "cuda", or "mps"
TORCH_DTYPE = "auto"  # "auto", "float16", "bfloat16", or "float32"

# Optional. caption_augmented/captions.py가 만든 JSONL cache를 지정하세요.
CAPTION_CACHE_PATH = Path("outputs/caption_augmented/train_captions.jsonl")

LOW_RELEVANCE_THRESHOLD = 0.05
PREVIEW_WIDTH = 260

In [ ]:
import math
from typing import Sequence

import pandas as pd
import torch
from IPython.display import Markdown, display
from PIL import Image
from transformers import AutoModel, AutoProcessor

from src.data_filtering.quality import load_caption_cache
from src.data_utils import INPUT_COLUMNS, image_paths_for_row, read_csv


def resolve_device(device: str) -> str:
    if device != "auto":
        return device
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def resolve_dtype(dtype_name: str, device: str):
    if dtype_name == "auto":
        return torch.float16 if device == "cuda" else None
    return {
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
        "float32": torch.float32,
    }[dtype_name]


def normalize(tensor: torch.Tensor) -> torch.Tensor:
    return tensor / tensor.norm(p=2, dim=-1, keepdim=True).clamp_min(1e-12)


def combined_score(image_score, caption_score):
    values = [value for value in [image_score, caption_score] if value is not None]
    return min(values) if values else None


device = resolve_device(DEVICE)
dtype = resolve_dtype(TORCH_DTYPE, device)
model_kwargs = {"torch_dtype": dtype} if dtype is not None else {}

print(f"device={device}, dtype={dtype}")
processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
model = AutoModel.from_pretrained(SIGLIP_MODEL, **model_kwargs).to(device)
model.eval()
print(f"loaded {SIGLIP_MODEL}")

In [ ]:
df = read_csv(DATA_DIR / f"{SPLIT}.csv")
matches = df[df["Id"].astype(str) == str(DATASET_ID)]
if matches.empty:
    raise ValueError(f"Id={DATASET_ID!r} not found in {DATA_DIR / f'{SPLIT}.csv'}")

row = matches.iloc[0]
image_dir = DATA_DIR / SPLIT
image_paths = image_paths_for_row(row, image_dir)

caption_cache = load_caption_cache(CAPTION_CACHE_PATH) if CAPTION_CACHE_PATH and CAPTION_CACHE_PATH.exists() else {}
captions = []
for image_index, column in enumerate(INPUT_COLUMNS, start=1):
    image_name = str(row[column])
    captions.append(caption_cache.get((str(row["Id"]), image_index, image_name)))

metadata = {
    "Id": row["Id"],
    "Sentence": row["Sentence"],
}
if "Answer" in row:
    metadata["Answer"] = row["Answer"]
if "No_ordering" in row:
    metadata["No_ordering"] = row["No_ordering"]

display(pd.DataFrame([metadata]).T.rename(columns={0: "value"}))

for image_index, image_path in enumerate(image_paths, start=1):
    display(Markdown(f"### Input_{image_index}: `{image_path.name}`"))
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    new_height = max(1, int(height * PREVIEW_WIDTH / width))
    display(image.resize((PREVIEW_WIDTH, new_height)))
    caption = captions[image_index - 1]
    if caption:
        display(Markdown(f"caption: {caption}"))
    else:
        display(Markdown("caption: *(missing)*"))

In [ ]:
def score_images(sentence: str, paths: Sequence[Path]) -> list[float | None]:
    scores: list[float | None] = [None] * len(paths)
    valid_items = [(index, path) for index, path in enumerate(paths) if path.exists()]
    if not sentence.strip() or not valid_items:
        return scores

    images = []
    valid_indices = []
    for index, path in valid_items:
        with Image.open(path) as image:
            images.append(image.convert("RGB"))
        valid_indices.append(index)

    inputs = processor(
        text=[sentence],
        images=images,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.sigmoid(outputs.logits_per_image)[:, 0].detach().cpu().tolist()

    for index, score in zip(valid_indices, probabilities, strict=True):
        scores[index] = float(score)
    return scores


def encode_texts(texts: Sequence[str]) -> torch.Tensor:
    inputs = processor(
        text=list(texts),
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        features = model.get_text_features(**inputs)
    return normalize(features.float())


def score_captions(sentence: str, caption_values: Sequence[str | None]) -> list[float | None]:
    scores: list[float | None] = [None] * len(caption_values)
    valid_items = [
        (index, str(caption).strip())
        for index, caption in enumerate(caption_values)
        if caption is not None and str(caption).strip()
    ]
    if not sentence.strip() or not valid_items:
        return scores

    sentence_features = encode_texts([sentence])
    caption_features = encode_texts([caption for _, caption in valid_items])
    similarities = (caption_features @ sentence_features.T)[:, 0].detach().cpu().tolist()
    for (index, _), score in zip(valid_items, similarities, strict=True):
        scores[index] = float(score)
    return scores


sentence = str(row["Sentence"])
image_scores = score_images(sentence, image_paths)
caption_scores = score_captions(sentence, captions)
combined_scores = [combined_score(image_score, caption_score) for image_score, caption_score in zip(image_scores, caption_scores)]

result = pd.DataFrame(
    {
        "image_index": [1, 2, 3, 4],
        "input_column": INPUT_COLUMNS,
        "image_file": [path.name for path in image_paths],
        "image_sentence_siglip_prob": image_scores,
        "caption": captions,
        "caption_sentence_text_cosine": caption_scores,
        "combined_relevance": combined_scores,
        "below_threshold": [score is not None and score <= LOW_RELEVANCE_THRESHOLD for score in combined_scores],
    }
)
display(result)

In [ ]:
low = result[result["below_threshold"]]
if low.empty:
    display(Markdown(f"No frames are below threshold `{LOW_RELEVANCE_THRESHOLD}`."))
else:
    display(Markdown(f"Frames below threshold `{LOW_RELEVANCE_THRESHOLD}`:"))
    display(low[["image_index", "image_file", "combined_relevance", "image_sentence_siglip_prob", "caption_sentence_text_cosine", "caption"]])